In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from Code.ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
from dotenv import load_dotenv
import os 
from openai import OpenAI

load_dotenv()
or_client = OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key= os.getenv("OPENROUTER_API_KEY")
)

In [5]:
from Code.evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=or_client,
)

In [6]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

"Yes, you can still sign up/join. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions. \n\nAdditionally, based on the course's general guidelines, you don't strictly need to complete a formal registration—you can simply start learning and submitting homework (as long as the submission form is still open) without registering beforehand."

In [7]:
assistant.total_cost()

0.00015475000000000002

In [8]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [9]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found out about the class—can I still sign up?',
 'answer_llm': "Yes, you can still sign up/join. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions. \n\nAdditionally, based on the course's general guidelines, you don't strictly need to complete a formal registration—you can simply start learning and submitting homework (as long as the submission form is still open) without registering beforehand.",
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [10]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [13]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found out about the class—can I still sign up?',
 'answer_llm': 'Yes, you can still join! According to the course FAQ:\n\n> "I just discovered the course. Can I still join?  \n> A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions."\n\nAdditionally, registration isn\'t strictly required—you can simply start learning and submitting homework (as long as the submission form is open) without formal sign-up. Just make sure to submit your project before submissions close if you want a certificate.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [15]:
assistant.reset_usage()

In [16]:
from concurrent.futures import ThreadPoolExecutor
from Code.evaluation_utils import map_progress

In [17]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/517 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1783555200000'}, 'provider_name': None}}, 'user_id': 'user_3FJjSghQkHJd5qcABXoDOzBYMAL'}

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [ ]:
assistant.total_cost()

In [ ]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)